# Construção do Banco Analítico — Projeto VigiMed  
# Analytical Database Construction — VigiMed Project

---

## 📘 Descrição | Description

**PT:**  
Este notebook reconstrói o banco analítico utilizado nas análises do estudo
a partir dos dados padronizados disponibilizados publicamente no repositório
Harvard Dataverse.

O objetivo é permitir a **reprodutibilidade completa** das análises,
possibilitando que outros pesquisadores reconstruam a base de dados e
reproduzam os resultados de forma independente.

**EN:**  
This notebook reconstructs the analytical database used in the study from
standardized data publicly available in the Harvard Dataverse repository.

Its purpose is to ensure **full reproducibility** of the analyses,
allowing other researchers to independently rebuild the database and
reproduce the results.

---

## 🔄 Fluxo executado | Workflow

**PT — O notebook executa automaticamente:**

1. Instalação dos programas necessários
2. Upload dos arquivos obtidos no Dataverse
3. Organização automática dos dados
4. Criação das tabelas no banco DuckDB
5. Consolidação dos dados em uma base analítica final
6. Exportação do dataset final

**EN — The notebook automatically performs:**

1. Installation of required software packages
2. Upload of files obtained from Dataverse
3. Automatic organization of data files
4. Creation of DuckDB database tables
5. Consolidation of data into a final analytical dataset
6. Export of the final dataset

---

## ▶️ Instruções de uso | Usage instructions

**PT:**

1. Baixe todos os arquivos `.parquet` disponíveis no Dataverse.
2. Abra este notebook no Google Colab.
3. Faça upload dos arquivos quando solicitado.
4. Execute todas as células do notebook.

Dependendo do tamanho dos dados, o processo pode levar alguns minutos.

Ao final, o banco analítico será reconstruído automaticamente.

**EN:**

1. Download all `.parquet` files available in Dataverse.
2. Open this notebook using Google Colab.
3. Upload the files when requested.
4. Run all notebook cells.

Depending on data size, execution may take a few minutes.

The analytical database will be automatically reconstructed.

---

## 📦 Sobre os arquivos de dados | About data files

**PT:**  
Os arquivos no formato `.parquet` são arquivos de dados otimizados para
análise e utilizados apenas como fonte de dados. Não é necessário abrir
ou modificar esses arquivos manualmente.

**EN:**  
Files in `.parquet` format are optimized for data analysis and used only
as data sources. Users do not need to open or modify them manually.

---

## 📊 Fonte dos dados | Data source

**PT:**  
O banco de dados padronizado utilizado neste notebook está publicado no
repositório Harvard Dataverse.

Dataset DOI: **10.xxxx/xxxxxx**

**EN:**  
The standardized database used in this notebook is publicly available in
the Harvard Dataverse repository.

Dataset DOI: **10.xxxx/xxxxxx**

---

## 🔒 Uso e citação | Usage and citation

**PT:**  
Este projeto e o banco de dados são públicos e de acesso aberto. Caso
utilize este notebook ou os dados em análises ou publicações, por favor
cite o projeto e o DOI do dataset.

**EN:**  
This project and database are publicly available. If you use this notebook
or dataset in analyses or publications, please cite the project and the
dataset DOI.

---

## 📰 Referência | Reference

Jacoby AC, Barnabé SG, Matsuda MA, Mendoza MR, Cazella SC.  
**Adverse drug reaction records from the Brazilian pharmacovigilance
system (VigiMed), 2018–2025.** 2026.  
Available at: [Insert DOI or repository link].

---

## 📧 Contato | Contact

Ana Carolina Jacoby  
Email: anacarolinajacoby0@gmail.com


## 1- Instalação das dependências | Installing dependencies

**PT:**  
Esta etapa instala automaticamente os programas necessários para executar
o notebook.

**EN:**  
This step automatically installs the required software packages needed to
run the notebook.


In [ ]:
# Instala bibliotecas necessárias
!pip install duckdb pandas pyarrow --quiet


In [ ]:
print("Instalando dependências...")

!pip install duckdb pandas pyarrow --quiet

print("Dependências instaladas com sucesso.")


## 2- Importação das bibliotecas | Importing libraries

**PT:**  
Carrega as bibliotecas utilizadas para manipulação dos dados e construção
do banco analítico.

**EN:**  
Loads the libraries used for data processing and database construction.


In [ ]:
import duckdb
import pandas as pd
from pathlib import Path
import os

print("Bibliotecas carregadas com sucesso.")


## 3- Upload dos arquivos de dados | Data upload

**PT:**  
Selecione **todos os arquivos `.parquet` baixados do Dataverse**.

Você pode selecionar vários arquivos ao mesmo tempo.

⚠️ O notebook não funcionará se algum arquivo estiver faltando.

**EN:**  
Select **all `.parquet` files downloaded from Dataverse**.

You can upload multiple files at once.

⚠️ The notebook will not run correctly if any file is missing.


In [ ]:
from google.colab import files

print("Selecione os arquivos .parquet para upload:")

uploaded = files.upload()

print(f"{len(uploaded)} arquivos enviados com sucesso.")


## 4- Verificação dos arquivos enviados | Uploaded files check

**PT:**  
Esta etapa verifica se todos os arquivos necessários foram enviados.

Caso algum arquivo esteja faltando, o notebook será interrompido para
evitar erros nas etapas seguintes.

**EN:**  
This step checks whether all required files were uploaded.

If any file is missing, execution stops to avoid errors in later steps.


In [ ]:
expected_files = [
    "dim_atc",
    "fat_medicamentos",
    "dim_notificacoes",
    "fat_reacoes",
    "dim_soc_llt"
]

missing = [f for f in expected_files if not any(f in name for name in uploaded)]

if missing:
    raise ValueError(
        f"""
Arquivos faltando: {missing}

Faça upload novamente incluindo todos os arquivos necessários.

Missing files: {missing}
Please upload all required files and try again.
"""
    )

print("Todos os arquivos necessários foram enviados.")


## 5- Organização automática dos arquivos | Automatic file organization

**PT:**  
Os arquivos enviados serão organizados automaticamente na estrutura de
pastas esperada pelo banco de dados.

Nenhuma ação adicional é necessária.

**EN:**  
Uploaded files will be automatically organized into the folder structure
expected by the database.

No additional action is required.


In [ ]:
DATA_DIR = Path("data/03_gold")

# Criar estrutura de pastas
folders = [
    "dim_atc",
    "fat_medicamentos",
    "dim_notificacoes",
    "fat_reacoes",
    "dim_soc_llt"
]

for folder in folders:
    (DATA_DIR / folder).mkdir(parents=True, exist_ok=True)

print("Estrutura de pastas criada.")

# Organizar arquivos
for file in uploaded.keys():

    if "dim_atc" in file:
        os.rename(file, DATA_DIR / "dim_atc/dim_atc.parquet")

    elif "fat_medicamentos" in file:
        os.rename(file, DATA_DIR / "fat_medicamentos/fat_medicamentos.parquet")

    elif "dim_notificacoes" in file:
        os.rename(file, DATA_DIR / "dim_notificacoes/dim_notificacoes.parquet")

    elif "fat_reacoes" in file:
        os.rename(file, DATA_DIR / "fat_reacoes/fat_reacoes.parquet")

    elif "dim_soc_llt" in file:
        os.rename(file, DATA_DIR / "dim_soc_llt/dim_soc_llt.parquet")

print("Arquivos organizados com sucesso.")


## 6- Definição dos caminhos dos dados | Data path definition

**PT:**  
Nesta etapa definimos os caminhos dos arquivos que serão utilizados para
criar as tabelas do banco de dados.

**EN:**  
This step defines the file paths used to create the database tables.


In [ ]:
path = DATA_DIR

path_atc = str(path / 'dim_atc/dim_atc.parquet')
path_fat_medicamentos = str(path / 'fat_medicamentos/fat_medicamentos.parquet')
path_dim_notificacoes = str(path / 'dim_notificacoes/dim_notificacoes.parquet')
path_fat_reacoes = str(path / 'fat_reacoes/fat_reacoes.parquet')
path_dim_soc_llt = str(path / 'dim_soc_llt/dim_soc_llt.parquet')


In [ ]:
path = DATA_DIR

path_atc = str(path / "dim_atc/dim_atc.parquet")
path_fat_medicamentos = str(path / "fat_medicamentos/fat_medicamentos.parquet")
path_dim_notificacoes = str(path / "dim_notificacoes/dim_notificacoes.parquet")
path_fat_reacoes = str(path / "fat_reacoes/fat_reacoes.parquet")
path_dim_soc_llt = str(path / "dim_soc_llt/dim_soc_llt.parquet")

print("Caminhos dos arquivos definidos com sucesso.")


## 7- Inicialização do banco de dados | Database initialization

**PT:**  
Nesta etapa criamos o banco de dados analítico utilizando DuckDB.

DuckDB é um banco leve executado dentro do notebook e não requer
instalação adicional no computador do usuário.

**EN:**  
This step creates the analytical database using DuckDB.

DuckDB is a lightweight database running inside the notebook and does not
require additional installation on the user's computer.


In [ ]:
import duckdb

con = duckdb.connect("vigimed.duckdb")

print("Banco DuckDB inicializado com sucesso.")



## 8- Criação das tabelas do banco | Database table creation

**PT:**  
Nesta etapa os arquivos de dados são carregados e transformados em tabelas
dentro do banco analítico.

Cada conjunto de dados gera uma tabela utilizada nas análises seguintes.

**EN:**  
In this step, data files are loaded and converted into tables inside the
analytical database.

Each dataset becomes a table used in subsequent analyses.


In [ ]:
print("Criando tabela dim_atc...")
con.execute(f"""
CREATE OR REPLACE TABLE dim_atc AS
SELECT * FROM read_parquet('{path_atc}')
""")

print("Criando tabela fat_medicamentos...")
con.execute(f"""
CREATE OR REPLACE TABLE fat_medicamentos AS
SELECT * FROM read_parquet('{path_fat_medicamentos}')
""")

print("Criando tabela dim_notificacoes...")
con.execute(f"""
CREATE OR REPLACE TABLE dim_notificacoes AS
SELECT * FROM read_parquet('{path_dim_notificacoes}')
""")

print("Criando tabela fat_reacoes...")
con.execute(f"""
CREATE OR REPLACE TABLE fat_reacoes AS
SELECT * FROM read_parquet('{path_fat_reacoes}')
""")

print("Criando tabela dim_soc_llt...")
con.execute(f"""
CREATE OR REPLACE TABLE dim_soc_llt AS
SELECT * FROM read_parquet('{path_dim_soc_llt}')
""")

print("Todas as tabelas foram criadas com sucesso.")


## 9- Construção da base analítica | Analytical dataset construction

**PT:**  
Nesta etapa os dados das diferentes tabelas são combinados para formar a
base analítica consolidada utilizada nas análises subsequentes.

**EN:**  
In this step, data from different tables are combined to form the final
analytical dataset used in subsequent analyses.


In [ ]:
print("Construindo base analítica...")

con.execute("""
CREATE OR REPLACE TABLE fat_analitica AS
SELECT
    fm.id_notificacao,
    fm.id_medicamento,
    fr.reacao,
    dn.ano_notificacao
FROM fat_medicamentos fm
JOIN fat_reacoes fr
    ON fm.id_notificacao = fr.id_notificacao
JOIN dim_notificacoes dn
    ON fm.id_notificacao = dn.id_notificacao
""")

print("Base analítica criada com sucesso.")


## 10- Exportação da base analítica | Export analytical dataset

**PT:**  
A base analítica consolidada é exportada em formato `.parquet` para ser
utilizada nas etapas seguintes de análise.

**EN:**  
The consolidated analytical dataset is exported in `.parquet` format for
use in subsequent analysis steps.


In [ ]:
print("Exportando base analítica...")

con.execute("""
COPY fat_analitica
TO 'fat_analitica.parquet'
(FORMAT PARQUET);
""")

print("Arquivo fat_analitica.parquet gerado com sucesso.")


## 11- Download do dataset final | Download final dataset

**PT:**  
Faça o download do arquivo gerado para utilizá-lo nas próximas etapas de
análise.

**EN:**  
Download the generated file to use it in the next analysis steps.


In [ ]:
from google.colab import files

print("Preparando download do arquivo...")

files.download("fat_analitica.parquet")


## Processo concluído | Process completed


In [ ]:
print("""
Processo concluído com sucesso!

A base analítica foi construída e exportada.
Agora você pode seguir para os próximos notebooks de análise.
""")
